# KV Cache এবং Speculative Decoding

তিনটি demo:

1. একটি ক্ষুদ্র decoder-only Transformer (Phase 02 Lesson 6-এর মতোই recipe), সামান্য প্রশিক্ষিত, তারপর দুটি উপায়ে generation: একটি naive loop যা প্রতি ধাপে পুরো সিকোয়েন্সের উপর attention পুনরায় গণনা করে, এবং একটি প্রকৃত **KV-CACHED** loop যা K/V tensors cache করে পুনরায় ব্যবহার করে। আমরা যাচাই করি দুটোই byte-for-byte **অভিন্ন** token sequences তৈরি করে (cache হলো একটি বিশুদ্ধ speed optimization, approximation নয়), এবং প্রতিটি পদ্ধতি আসলে কতগুলো attended query-key pair গণনা করে তা গুনি — O(T) বনাম O(T^2)-per-step বৃদ্ধির প্যাটার্নটি সরাসরি সংখ্যায় দেখিয়ে (শুধু wall-clock নয়, যা এই toy scale-এ noise হতে পারে)।
2. একটি toy speculative-decoding simulation: একটি "draft" stand-in যা কিছু per-token সম্ভাব্যতা `p`-তে একটি "target" ground-truth sequence-এর সাথে একমত হয়, এবং একটি স্থির-দৈর্ঘ্যের সিকোয়েন্স তৈরি করতে প্রয়োজনীয় ব্যয়বহুল target-model call-সংখ্যার প্রকৃত হ্রাস মাপে।

**চালানোর নিয়ম:** উপরের দিক থেকে নিচের দিকে প্রতিটি cell চালান (runtime CPU-তে এক মিনিটেরও কম: একটি ক্ষুদ্র character corpus-এ ~200-ধাপের training run, যোগ কিছু ছোট simulation)। প্রথম সেকশনে কেবল setup (corpus, tokenizer, constants ও `get_batch`) আছে — সেখানে কোনো demo চলে না; 1 ও 2 নম্বর সেকশনের demo তাদের নিজ নিজ cell-এর শেষেই চলে।

In [ ]:
import math
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
random.seed(0)

## 0. Setup: একটি ক্ষুদ্র toy corpus ও character-level tokenizer

Phase 02 Lesson 6-এর মতোই recipe, deliberately ছোট রাখা যাতে প্রশিক্ষণ সেকেন্ডেই শেষ হয়। এতে model constants ও `get_batch` training-batch helper-ও সংজ্ঞায়িত আছে — এখানে কোনো demo চলে না, এটি কেবল পরের সেকশনগুলোর জন্য প্রস্তুতি।

In [ ]:
# ---------------------------------------------------------------------------
# 0. একটি ক্ষুদ্র toy corpus এবং character-level tokenizer (Phase 02 Lesson 6-এর
#    মতোই recipe, ছোট রাখা হয়েছে যাতে প্রশিক্ষণ সেকেন্ডে শেষ হয়)।
# ---------------------------------------------------------------------------

CORPUS = """
the quick brown fox jumps over the lazy dog.
the lazy dog barks at the quick brown fox.
the fox runs into the dark forest at night.
""".strip().lower()
CORPUS = (CORPUS + "\n") * 12

chars = sorted(set(CORPUS))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}


def encode(text):
    return [stoi[ch] for ch in text]


def decode(ids):
    return "".join(itos[i] for i in ids)


data = torch.tensor(encode(CORPUS), dtype=torch.long)

D_MODEL = 32
NUM_HEADS = 4
D_FF = 4 * D_MODEL
NUM_LAYERS = 2
BLOCK_SIZE = 48
BATCH_SIZE = 16
NUM_ITERS = 250
LEARNING_RATE = 3e-3


def get_batch(data, block_size, batch_size):
    max_start = len(data) - block_size - 1
    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([data[s:s + block_size] for s in starts])
    y = torch.stack([data[s + 1:s + 1 + block_size] for s in starts])
    return x, y

## 1. Decoder-only Transformer, ঐচ্ছিক KV cache-সহ — অভিন্ন output, কম redundant আটেনশন compute

একটি decoder-only Transformer-এর attention layer যা **ঐচ্ছিকভাবে** KV cache ব্যবহার করতে পারে। `cache=None` দিলে এটি Phase 02 Lesson 6-এর সাধারণ full-sequence causal self-attention হুবহু পুনরুৎপাদন করে; cache দিলে এটি শুধু নতুন যোগ করা token(s)-কে প্রসেস করে, cached K/V-এর বিরুদ্ধে attend করে। Demo-টি model-টিকে সংক্ষিপ্তভাবে প্রশিক্ষণ দেয়, তারপর দুটি উপায়ে (naive ও cached) generation চালায়, outputs-এর অভিন্নতা যাচাই করে, এবং প্রতিটি পদ্ধতিতে মোট attended query-key pairs-এর সংখ্যা গুনে — O(T) বনাম O(T^2)-per-step বৃদ্ধি প্রকৃত সংখ্যায় দেখায়।

In [ ]:
# ---------------------------------------------------------------------------
# 1. একটি decoder-only Transformer যার attention layer ঐচ্ছিকভাবে KV cache
#    ব্যবহার করতে পারে। cache=None দিলে Phase 02 Lesson 6-এর সাধারণ
#    full-sequence causal self-attention হুবহু পুনরুৎপাদিত হয়। cache দিলে
#    শুধু নতুন যোগ করা token(s)-ই প্রসেস হয়, cached K/V-এর বিরুদ্ধে attend করে।
# ---------------------------------------------------------------------------

class CausalSelfAttentionCache(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def _split_heads(self, t, batch, T):
        return t.view(batch, T, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, x, counter, cache=None):
        """x: (batch, T_new, d_model) -- cache সরবরাহ করলে শুধু NEW tokens
        (cached generation-এ T_new সাধারণত 1 হয়), অথবা cache None হলে এখানে
        এখন পর্যন্ত FULL sequence (naive, সব-পুনরায়-গণনা পথ)। ফেরত দেয়
        (output, updated_cache)।"""
        batch, T_new, _ = x.shape
        Q = self._split_heads(self.W_q(x), batch, T_new)
        K_new = self._split_heads(self.W_k(x), batch, T_new)
        V_new = self._split_heads(self.W_v(x), batch, T_new)

        if cache is not None:
            K = torch.cat([cache["k"], K_new], dim=2)
            V = torch.cat([cache["v"], V_new], dim=2)
        else:
            K, V = K_new, V_new
        T_k = K.shape[2]

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)   # (batch, heads, T_new, T_k)
        if cache is None:
            # No cache মানে x-এ এখন পর্যন্ত WHOLE sequence আছে -> তাই স্বাভাবিক
            # causal mask দরকার যাতে position i শুধু keys 0..i-তে attend করে।
            mask = torch.tril(torch.ones(T_new, T_k)).bool()
            scores = scores.masked_fill(~mask, float("-inf"))
            valid_pairs = T_new * (T_new + 1) // 2   # sum_{i=0}^{T_new-1} (i+1)
        else:
            # Cache-সহ, x-এ শুধু ব্র্যান্ড-new token(s) থাকে: cache-এ ইতিমধ্যে
            # থাকা প্রতিটি key (যোগ নতুনটি) causalভাবে বৈধ, কোনো mask দরকার নেই।
            valid_pairs = T_new * T_k

        counter["pairs"] += batch * self.num_heads * valid_pairs

        weights = F.softmax(scores, dim=-1)
        out = (weights @ V).transpose(1, 2).contiguous().view(batch, T_new, self.num_heads * self.d_k)
        return self.W_o(out), {"k": K, "v": V}


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttentionCache(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x, counter, cache=None):
        attn_out, new_cache = self.attn(self.ln1(x), counter, cache)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x, new_cache


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, block_size):
        super().__init__()
        self.block_size = block_size
        self.num_layers = num_layers
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([DecoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.output_head = nn.Linear(d_model, vocab_size)

    def forward_full(self, token_ids, counter):
        """Naive পথ: পুরো সিকোয়েন্সের উপর causal self-attention পুনরায় গণনা করে,
        ঠিক Phase 02 Lesson 6-এর forward pass, কোনো cache ছাড়া।"""
        batch, T = token_ids.shape
        positions = torch.arange(T)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        for block in self.blocks:
            x, _ = block(x, counter, cache=None)
        x = self.final_norm(x)
        return self.output_head(x)   # (batch, T, vocab_size)

    def forward_step(self, new_token_id, position, cache_list, counter):
        """Cached পথ: শুধু একক নতুন token প্রসেস করে, cache_list (প্রতি layer-এ
        একটি {'k','v'} dict) পুনরায় ব্যবহার ও বর্ধিত করে।"""
        pos_tensor = torch.tensor([position])
        x = self.token_embedding(new_token_id) + self.position_embedding(pos_tensor)
        new_cache_list = []
        for block, cache in zip(self.blocks, cache_list):
            x, new_cache = block(x, counter, cache=cache)
            new_cache_list.append(new_cache)
        x = self.final_norm(x)
        return self.output_head(x), new_cache_list   # logits: (batch, 1, vocab_size)

    def train_loss(self, token_ids, targets):
        counter = {"pairs": 0}   # inference-cost মাপনার অংশ নয়
        logits = self.forward_full(token_ids, counter)
        return F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))


@torch.no_grad()
def generate_naive(model, start_ids, max_new_tokens, counter):
    """প্রতি ধাপে স্ক্র্যাচ থেকে WHOLE sequence-এর attention পুনরায় গণনা করে --
    Phase 02 Lesson 6-এর naive loop, attention কাজ গোনার জন্য instrumented।"""
    token_ids = start_ids.clone()
    for _ in range(max_new_tokens):
        context = token_ids[:, -model.block_size:]
        logits = model.forward_full(context, counter)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)   # greedy: deterministic
        token_ids = torch.cat([token_ids, next_token], dim=1)
    return token_ids


@torch.no_grad()
def generate_cached(model, start_ids, max_new_tokens, counter):
    """Prompt-কে এক সময়ে একটি token করে খাওয়িয়ে KV cache তৈরি করে, তারপর
    প্রতিটি ধাপে cache-কে একটি নতুন token বাড়িয়ে generation চালায় -- আগের
    positions-এর কোনো পুনরায় গণনা নেই।"""
    batch, T0 = start_ids.shape
    cache_list = [None] * model.num_layers
    token_ids = start_ids.clone()
    logits = None
    for pos in range(T0):
        tok = start_ids[:, pos:pos + 1]
        logits, cache_list = model.forward_step(tok, pos, cache_list, counter)
    for _ in range(max_new_tokens):
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        token_ids = torch.cat([token_ids, next_token], dim=1)
        pos = token_ids.shape[1] - 1
        logits, cache_list = model.forward_step(next_token, pos, cache_list, counter)
    return token_ids


def kv_cache_demo():
    print("=" * 70)
    print("1. KV CACHE: IDENTICAL OUTPUT, LESS REDUNDANT ATTENTION COMPUTE")
    print("=" * 70)

    model = MiniGPT(vocab_size, D_MODEL, NUM_HEADS, D_FF, NUM_LAYERS, BLOCK_SIZE)
    print(f"Tiny decoder-only model: {sum(p.numel() for p in model.parameters()):,} parameters, "
          f"{NUM_LAYERS} layers, {NUM_HEADS} heads")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    print(f"\nTraining briefly ({NUM_ITERS} steps) so generation isn't pure noise...")
    for step in range(1, NUM_ITERS + 1):
        x, y = get_batch(data, BLOCK_SIZE, BATCH_SIZE)
        loss = model.train_loss(x, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 50 == 0 or step == 1:
            print(f"  step {step:4d}  loss = {loss.item():.4f}")
    model.eval()

    start_ids = torch.tensor([encode("the ")], dtype=torch.long)
    max_new_tokens = 40

    counter_naive = {"pairs": 0}
    tokens_naive = generate_naive(model, start_ids, max_new_tokens, counter_naive)

    counter_cached = {"pairs": 0}
    tokens_cached = generate_cached(model, start_ids, max_new_tokens, counter_cached)

    identical = torch.equal(tokens_naive, tokens_cached)
    print(f"\nNaive generation output:  {decode(tokens_naive[0].tolist())!r}")
    print(f"Cached generation output: {decode(tokens_cached[0].tolist())!r}")
    print(f"\nOutputs are byte-for-byte IDENTICAL: {identical}")
    assert identical, "KV cache changed the output -- this would be a bug in the cache logic!"
    print("-> Confirms the KV cache is a pure speed optimization: same weights, same greedy")
    print("   decoding rule, same tokens out -- caching never changes WHAT the model computes,")
    print("   only how much redundant work it takes to compute it.")

    print(f"\nTotal attended query-key pairs (summed over all layers, heads, and generation steps):")
    print(f"  naive (recompute every step):  {counter_naive['pairs']:>10,}")
    print(f"  KV-cached:                     {counter_cached['pairs']:>10,}")
    print(f"  ratio (naive / cached):        {counter_naive['pairs'] / counter_cached['pairs']:>10.1f}x")

    # বৃদ্ধির প্যাটার্ন সরাসরি দেখানো: একটি দীর্ঘতর generation-এর per-step
    # attended-pair counts, শুধু একটি layer/head-এর কাজের worth-এ বিচ্ছিন্ন করে
    # যাতে O(T) বনাম O(T^2)-per-step আকার ধ্রুবক layer/head multiplier-এ
    # অস্পষ্ট না হয়।
    print("\nPer-step attended-pair counts for a single layer+head (context length T -> pairs")
    print("computed AT THAT STEP), showing the growth pattern as generation proceeds:")
    print(f"{'context length T':>18}{'naive: T*(T+1)/2':>22}{'cached: T':>14}")
    for T in [5, 10, 20, 40, 80]:
        naive_pairs_this_step = T * (T + 1) // 2
        cached_pairs_this_step = T
        print(f"{T:>18}{naive_pairs_this_step:>22,}{cached_pairs_this_step:>14,}")
    print("\n-> Naive per-step cost grows QUADRATICALLY with context length (it re-scores the")
    print("   entire prefix every time); cached per-step cost grows only LINEARLY (one new")
    print("   query against the existing keys). That per-step gap is exactly why the KV")
    print("   cache is the single most important systems optimization for long generations.")


kv_cache_demo()

## 2. Speculative decoding: draft প্রস্তাব করে, target একটি সমান্তরাল call-এ যাচাই করে

একটি toy Monte Carlo simulation: একটি "draft" model প্রতি রাউন্ডে `draft_tokens_per_round` টি প্রার্থী token প্রস্তাব করে, প্রতিটি স্বাধীনভাবে `draft_agreement_p` সম্ভাব্যতায় target-এর প্রকৃত পরবর্তী token-এর সাথে মেলে। Target model পুরো ব্যাচটি **একটিই** forward pass-এ যাচাই করে — গৃহীত হয় দীর্ঘতম সঠিক prefix, তারপর target নিজেই পরের token সরবরাহ করে। মাপা হয় একটি স্থির-দৈর্ঘ্যের সিকোয়েন্স তৈরি করতে প্রয়োজনীয় গড় ব্যয়বহুল target-model call-সংখ্যা, কয়েকটি agreement rate-এর জন্য।

In [ ]:
# ---------------------------------------------------------------------------
# 2. Speculative decoding: draft প্রস্তাব করে, target একটি সমান্তরাল call-এ
#    যাচাই করে, গৃহীত হয় দীর্ঘতম সঠিক prefix। একটি স্থির-দৈর্ঘ্যের target
#    ground-truth sequence-এর উপর Monte Carlo simulation হিসেবে মাপা হয়।
# ---------------------------------------------------------------------------

def simulate_speculative_decoding(seq_len, draft_agreement_p, draft_tokens_per_round, num_trials=500):
    """Speculative decoding ব্যবহার করে seq_len দৈর্ঘ্যের একটি সিকোয়েন্স তৈরি
    করতে প্রয়োজনীয় Gড় NUMBER of expensive target-model calls ফেরত দেয়।

    প্রতি রাউন্ড: draft model `draft_tokens_per_round` টি প্রার্থী token প্রস্তাব
    করে; প্রতিটি প্রার্থী `draft_agreement_p` সম্ভাব্যতায় স্বাধীনভাবে target-এর
    প্রকৃত পরবর্তী token-এর সাথে মেলে (ছোট draft model বড় target model-ের সাথে
    কতটা ভালোভাবে aligned তার একটি stand-in)। Target model প্রার্থীদের পুরো
    ব্যাচ একটি একক forward pass-এ যাচাই করে (প্রতি রাউন্ডে একটি target-model
    call, একসাথে যত প্রার্থী-ই যাচাই করুক না কেন): দীর্ঘতম সঠিক prefix গৃহীত
    হয়, তারপর পরের রাউন্ড শুরুর আগে target নিজেই পরবর্তী token সরবরাহ করে
    (প্রথম ভুল guess-এর পরে correction, অথবা প্রতিটি প্রার্থী যদি সঠিক হতো
    তাহলে বিনামূল্যে একটি bonus token)।"""
    total_calls = 0
    for _ in range(num_trials):
        pos = 0
        calls = 0
        while pos < seq_len:
            k = min(draft_tokens_per_round, seq_len - pos)
            # K টি draft guess-এর কতগুলো সঠিক, বাম থেকে ডানে, প্রথম ভুলটির
            # আগ পর্যন্ত (শুধু দীর্ঘতম সঠিক PREFIX)।
            accepted = 0
            for _ in range(k):
                if random.random() < draft_agreement_p:
                    accepted += 1
                else:
                    break
            calls += 1   # প্রতি রাউন্ডে ঠিক একটি target verification pass
            # accepted tokens সঠিক বলে নিশ্চিত; target এই রাউন্ডে আরও একটি
            # token সরবরাহ করে (guess ভুল হলে একটি correction, অথবা রাউন্ডের
            # প্রতিটি guess গৃহীত হলে একটি বিনামূল্যের bonus token)।
            pos += accepted + 1
        total_calls += calls
    return total_calls / num_trials


def speculative_decoding_demo():
    print("\n" + "=" * 70)
    print("2. SPECULATIVE DECODING: FEWER EXPENSIVE TARGET-MODEL CALLS")
    print("=" * 70)

    seq_len = 60
    draft_tokens_per_round = 4
    print(f"Generating a fixed sequence of length {seq_len}.")
    print(f"WITHOUT speculative decoding, the target model is called once per token:")
    print(f"  target-model calls = {seq_len} (always, by definition)\n")

    print(f"WITH speculative decoding (draft proposes {draft_tokens_per_round} tokens/round, "
          f"target verifies all {draft_tokens_per_round} in ONE call):")
    print(f"{'draft/target agreement p':>26}{'avg target calls':>20}{'reduction vs no-spec':>24}")
    for p in [0.3, 0.5, 0.7, 0.9]:
        avg_calls = simulate_speculative_decoding(seq_len, p, draft_tokens_per_round)
        reduction = (1 - avg_calls / seq_len) * 100
        print(f"{p:>26.1f}{avg_calls:>20.1f}{reduction:>23.1f}%")

    print("\n-> Every row generates the SAME sequence length, using the SAME target model")
    print("   verification rule, so output quality is unaffected -- only the low-agreement")
    print("   draft (p=0.3) barely beats plain one-token-at-a-time decoding, while a")
    print("   well-aligned draft (p=0.9) lets the target confirm several tokens per call,")
    print("   cutting the number of expensive target-model forward passes substantially.")
    print("   This is exactly why speculative decoding's speedup depends entirely on how")
    print("   well the cheap draft model's guesses track the expensive target model.")


speculative_decoding_demo()

In [ ]:
def main():
    kv_cache_demo()
    speculative_decoding_demo()


main()